# L4 — AI Guard Gateway Circuit Breaker

**Secure at the perimeter** — this notebook adds a Datadog AI Guard **circuit breaker** at the AgentCore
Gateway. When the runtime hook (enabled in Notebook 2) writes an `ABORT` LOCK to the risk table, a **REQUEST interceptor** here
denies every subsequent **specialist tool call** — those routed through the tool Gateway (`anycompany-agent-gateway`, authorized by the Cognito M2M service token) — until the lock's TTL expires. (Registry discovery/invocation flows through the Harness's separate IAM Gateway, `anycompany-harness-gateway`, and is out of scope for this breaker.)

The AI Guard hook is enabled in `2_aiguard_agent_protection.ipynb`; this notebook adds the perimeter control.
The kept PII-masking **RESPONSE** interceptor (`1_sensitive_data_masking.ipynb`) stays attached alongside it.

## What You'll Learn

- Why a single agent's verdict **cannot contain a multi-agent session**, and how a perimeter control closes that gap.
- How to attach a **REQUEST** interceptor to an AgentCore Gateway that already has a **RESPONSE** interceptor
  (max one of each — you must re-send both in `update_gateway`).
- How the interceptor derives the **same service `sub`** the runtime hook used, with no shared session id.
- The honest **scaling limitation** of a service-wide breaker and the production path to per-user scoping.

## Overview

`AIGuardAbortError` terminates only the *current* specialist invocation. The orchestrator (an AgentCore Harness,
with no AI Guard hook) sees a bare failed A2A call and may simply **re-delegate to another agent** — a
non-deterministic LLM decision that cannot be trusted to halt. The gateway breaker is the deterministic control:
once the L3 hook writes a LOCK on `ABORT`, this interceptor 403s **every specialist tool call** through the tool Gateway (`anycompany-agent-gateway`) until TTL. Registry discovery/invocation runs through the Harness's separate IAM Gateway and is outside this breaker's scope, so a locked session is still contained where it matters — the specialist can no longer perform order/refund actions.

This notebook: builds the interceptor Lambda + its own role (`dynamodb:GetItem`), appends its ARN to the Gateway
role's `InvokeLambdaTargets`, and attaches it at the **REQUEST** point via `update_gateway` — re-sending the kept
PII RESPONSE config so neither is dropped.

## Architecture

```
  Agent tool call  ──Bearer <fixed service JWT>──▶  AgentCore Gateway (CUSTOM_JWT)
                                                      │
        ┌─────────────────────────────────────────────┴──────────────────────────┐
        │  REQUEST interceptor  [NEW — AI Guard circuit breaker]                   │
        │    • method == tools/call ?                                              │
        │    • decode JWT sub from Authorization  (== the sub the L3 hook wrote)   │
        │    • GetItem AnyCompanyAgentRisk[principalId=sub]                         │
        │    • status==LOCKED → transformedGatewayResponse 403 (tool NEVER runs)   │
        │    • else → pass through                                                 │
        └──────────────────────────────────┬───────────────────────────────────────┘
                                            ▼ (if allowed)
                                        Tool Lambda ─▶ DynamoDB / KB
                                            │
        ┌───────────────────────────────────┴──────────────────────────────────────┐
        │  RESPONSE interceptor  [KEPT — Bedrock Guardrail PII masking]              │
        └────────────────────────────────────────────────────────────────────────────┘
```

**Two distinct IAM roles:** the interceptor Lambda's OWN role gets `dynamodb:GetItem`; the L3 Runtime role (which
writes the lock) got `dynamodb:PutItem` separately. The Gateway role gets `lambda:InvokeFunction` on the interceptor.

## Steps
1. Load configuration from SSM (gateway, risk table, the kept PII interceptor ARN).
2. Create the interceptor Lambda's execution role (`dynamodb:GetItem` on the risk table + logs).
3. Create/update the `anycompany_aiguard_interceptor` Lambda.
4. Append the interceptor ARN to the Gateway role's `InvokeLambdaTargets`.
5. Attach at the REQUEST point via `update_gateway` (re-sending the kept RESPONSE config).
6. Verify: seed a LOCK → tool call 403s, tool Lambda NOT invoked; sub matches.
7. Restore normal operation: delete the LOCK row so tool calls route again.
8. Generate AI Guard evaluations: invoke the Harness with benign prompts so evaluations appear in Datadog → AI Guard → Investigate.

> **Step 7b (testing aside):** globally disable/re-enable the breaker in real time via a
> `__breaker_config__` row in the risk table — no redeploy. Used only for workshop testing.


## License Details

Refer to the LICENSE file in the main folder for license details.

## Prerequisites

- `l4-security/2_aiguard_agent_protection.ipynb` completed — the risk table + `/anycompany/agentcore/risk_table` exist.
- `l4-security/1_sensitive_data_masking.ipynb` completed — the PII RESPONSE interceptor is attached (we re-send it).
- The Gateway exists (L3 `1_setup_resources.ipynb`).

> Note: `update_gateway` rejects an empty `interceptorConfigurations=[]` — we always re-send BOTH configs (the kept RESPONSE + the new REQUEST).

## Step 1: Load Configuration from SSM

In [1]:
import os, json, time, zipfile, io
import boto3

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
SSM_PREFIX = "/anycompany/agentcore"

ssm = boto3.client("ssm", region_name=REGION)
iam = boto3.client("iam")
lambda_client = boto3.client("lambda", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        if default is not None: return default
        raise

ACCOUNT_ID = sts.get_caller_identity()["Account"]
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_NAME = get_ssm(f"{SSM_PREFIX}/gateway_name")
GATEWAY_ROLE_ARN = get_ssm(f"{SSM_PREFIX}/gateway_role_arn")
COGNITO_USER_POOL_ID = get_ssm(f"{SSM_PREFIX}/cognito_user_pool_id")
COGNITO_CLIENT_ID = get_ssm(f"{SSM_PREFIX}/cognito_client_id")
GATEWAY_M2M_CLIENT_ID = get_ssm(f"{SSM_PREFIX}/gateway_m2m_client_id")
GATEWAY_SCOPE = get_ssm(f"{SSM_PREFIX}/gateway_scope")
RISK_TABLE = get_ssm(f"{SSM_PREFIX}/risk_table")
RISK_TABLE_ARN = f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{RISK_TABLE}"
print(f"Gateway {GATEWAY_ID}  RiskTable {RISK_TABLE}")

Gateway anycompany-agent-gateway-m1jnn7lvju  RiskTable AnyCompanyAgentRisk


## Step 2: Interceptor Lambda Execution Role

The interceptor reads the risk table via its **own** role — distinct from the L3 Runtime role. It needs only
`dynamodb:GetItem` on the risk table plus basic logs.

In [2]:
INTERCEPTOR_ROLE_NAME = "AnyCompanyAIGuardInterceptorLambdaRole"
INTERCEPTOR_LAMBDA_NAME = "anycompany_aiguard_interceptor"

trust = json.dumps({"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]})

perms = json.dumps({"Version": "2012-10-17", "Statement": [
    {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
     "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/lambda/{INTERCEPTOR_LAMBDA_NAME}:*"},
    {"Effect": "Allow", "Action": ["dynamodb:GetItem"], "Resource": RISK_TABLE_ARN}]})

try:
    r = iam.create_role(RoleName=INTERCEPTOR_ROLE_NAME, AssumeRolePolicyDocument=trust)
    INTERCEPTOR_ROLE_ARN = r["Role"]["Arn"]
    print(f"Created: {INTERCEPTOR_ROLE_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    INTERCEPTOR_ROLE_ARN = iam.get_role(RoleName=INTERCEPTOR_ROLE_NAME)["Role"]["Arn"]
    print(f"Exists: {INTERCEPTOR_ROLE_ARN}")

iam.put_role_policy(RoleName=INTERCEPTOR_ROLE_NAME, PolicyName="AIGuardInterceptorPerms", PolicyDocument=perms)
print("Permissions attached.")
time.sleep(10)  # IAM propagation

Created: arn:aws:iam::359894365202:role/AnyCompanyAIGuardInterceptorLambdaRole
Permissions attached.


## Step 3: Create the REQUEST Interceptor Lambda

The handler decodes the JWT `sub` from the (gateway-validated) Authorization header — the **same** value the L3
hook wrote — does one tight `GetItem`, and returns a 403 `transformedGatewayResponse` when the principal is LOCKED.
Idempotent and stateless; the row's TTL auto-releases the breaker.

In [3]:
INTERCEPTOR_CODE = '''import base64
import json
import logging
import os
import time

import boto3

logger = logging.getLogger()
logger.setLevel(logging.INFO)

ddb = boto3.client("dynamodb")
RISK_TABLE = os.environ.get("RISK_TABLE", "")

# Sentinel principalId for the GLOBAL breaker on/off config row (never a real JWT sub).
BREAKER_CONFIG_KEY = "__breaker_config__"


def _breaker_active():
    """Return True if the gateway circuit breaker is globally ENABLED.

    Real-time on/off read from the SAME risk table (no SSM, no cache) via a strongly
    consistent read, so a flip is seen on the very next tool call. Fail-safe: any
    missing row or read error => ACTIVE (enforce). Disabling therefore requires an
    explicit ``enabled=false`` row and can never happen by deletion, absence, or a
    transient error.
    """
    try:
        item = ddb.get_item(
            TableName=RISK_TABLE,
            Key={"principalId": {"S": BREAKER_CONFIG_KEY}},
            ConsistentRead=True,
        ).get("Item")
    except Exception as exc:  # noqa: BLE001 - fail-safe: enforce on error
        logger.error(f"Breaker config read failed: {exc}; defaulting to ENABLED.")
        return True
    if not item:
        return True
    return item.get("enabled", {}).get("BOOL", True) is not False


def _decode_sub(authorization_header: str) -> str:
    """Decode (NOT verify) the JWT `sub` from an 'Authorization: Bearer <jwt>' header.

    Safe because the Gateway's CUSTOM_JWT authorizer has already validated the token on
    the request path; we only read the subject claim to key the risk lookup. Mirrors the
    runtime hook's decode exactly so both sides derive the SAME service `sub`.
    """
    if not authorization_header:
        return ""
    try:
        token = authorization_header.split(" ", 1)[1] if " " in authorization_header else authorization_header
        payload_b64 = token.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        payload = json.loads(base64.urlsafe_b64decode(payload_b64))
        return payload.get("sub", "")
    except Exception as exc:  # noqa: BLE001
        logger.warning(f"Could not decode sub from Authorization header: {exc}")
        return ""


def _passthrough(body):
    return {"interceptorOutputVersion": "1.0", "mcp": {"transformedGatewayRequest": {"body": body}}}


def _blocked_403(request_id, reason="Blocked by AI Guard security circuit breaker"):
    return {
        "interceptorOutputVersion": "1.0",
        "mcp": {
            "transformedGatewayResponse": {
                "statusCode": 403,
                "headers": {"Content-Type": "application/json"},
                "body": {
                    "jsonrpc": "2.0",
                    "id": request_id,
                    "error": {"code": -32002, "message": reason},
                },
            }
        },
    }


def lambda_handler(event, context=None):
    mcp = event.get("mcp", {})
    req = mcp.get("gatewayRequest", {})
    body = req.get("body", {})

    # Only gate tool calls; everything else (initialize, tools/list, ...) passes through.
    if body.get("method") != "tools/call":
        return _passthrough(body)

    # GLOBAL kill-switch (real-time, read from the risk table). When an operator has
    # written an `enabled=false` config row, bypass the breaker entirely: skip the lock
    # lookup and pass through. Logged loudly so a disabled breaker is never silent.
    if not _breaker_active():
        logger.warning("AI Guard gateway breaker GLOBALLY DISABLED via risk-table config; passing through.")
        return _passthrough(body)

    # Derive the service principal from the bearer token (requires passRequestHeaders=true).
    auth = req.get("headers", {}).get("Authorization", "") or req.get("headers", {}).get("authorization", "")
    sub = _decode_sub(auth)
    if not sub:
        # Cannot identify the principal — fail OPEN here (the runtime hook is the primary
        # control; the breaker is defense-in-depth) but log loudly. Flip to fail-closed
        # in production if the gateway guarantees a decodable token.
        logger.warning("No decodable sub on tools/call; passing through (breaker cannot key).")
        return _passthrough(body)

    try:
        item = ddb.get_item(
            TableName=RISK_TABLE,
            Key={"principalId": {"S": sub}},
            # Strongly consistent so a Step 7 lock delete is visible on the very next
            # tool call (an eventually-consistent read can return a just-deleted LOCK
            # for a few seconds -> spurious 403 in Step 8).
            ConsistentRead=True,
        ).get("Item")
    except Exception as exc:  # noqa: BLE001
        logger.error(f"Risk table GetItem failed: {exc}; passing through.")
        return _passthrough(body)

    if item and item.get("status", {}).get("S") == "LOCKED":
        # Read-time expiry: DynamoDB TTL deletion is best-effort and can lag hours, so do NOT
        # rely on it to release the breaker. Honor the lock's own ttl and treat an expired lock
        # as released (TTL reclaims the row later). This makes LOCK_TTL_SECONDS actually hold.
        try:
            lock_ttl = int(item.get("ttl", {}).get("N", "0"))
        except (ValueError, TypeError):
            lock_ttl = 0
        if lock_ttl and int(time.time()) >= lock_ttl:
            logger.info(f"AI Guard lock for principal={sub} expired (ttl={lock_ttl}); passing through.")
            return _passthrough(body)
        logger.warning(f"AI Guard circuit breaker OPEN for principal={sub}: 403 tools/call")
        return _blocked_403(body.get("id"))

    return _passthrough(body)
'''

# Package + create/update the function.
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("lambda_function.py", INTERCEPTOR_CODE)
buf.seek(0)
zip_bytes = buf.read()

try:
    resp = lambda_client.create_function(
        FunctionName=INTERCEPTOR_LAMBDA_NAME,
        Runtime="python3.12",
        Role=INTERCEPTOR_ROLE_ARN,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=10,
        Environment={"Variables": {"RISK_TABLE": RISK_TABLE}},
    )
    INTERCEPTOR_LAMBDA_ARN = resp["FunctionArn"]
    print(f"Created: {INTERCEPTOR_LAMBDA_ARN}")
except lambda_client.exceptions.ResourceConflictException:
    lambda_client.update_function_code(FunctionName=INTERCEPTOR_LAMBDA_NAME, ZipFile=zip_bytes)
    # update_function_code is asynchronous - wait for it to finish before the next
    # update op, otherwise update_function_configuration races with
    # "An update is in progress for resource".
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
    lambda_client.update_function_configuration(
        FunctionName=INTERCEPTOR_LAMBDA_NAME, Environment={"Variables": {"RISK_TABLE": RISK_TABLE}})
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
    INTERCEPTOR_LAMBDA_ARN = lambda_client.get_function(FunctionName=INTERCEPTOR_LAMBDA_NAME)["Configuration"]["FunctionArn"]
    print(f"Updated: {INTERCEPTOR_LAMBDA_ARN}")

ssm.put_parameter(Name=f"{SSM_PREFIX}/aiguard_interceptor_arn", Value=INTERCEPTOR_LAMBDA_ARN, Type="String", Overwrite=True)

Created: arn:aws:lambda:us-west-2:359894365202:function:anycompany_aiguard_interceptor


{'Version': 1,
 'Tier': 'Standard',
 'ResponseMetadata': {'RequestId': 'b97a9892-c091-4052-8ad0-27d7d35b159b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Wed, 07 Oct 2026 22:00:04 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '31',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'b97a9892-c091-4052-8ad0-27d7d35b159b',
   'cache-control': 'no-store'},
  'RetryAttempts': 0}}

## Step 4: Allow the Gateway Role to Invoke the Interceptor

Append the interceptor ARN to the Gateway role's `InvokeLambdaTargets` inline statement — the same pattern the PII
notebook uses (idempotent: skip if already present).

In [4]:
GATEWAY_ROLE_NAME = GATEWAY_ROLE_ARN.split("/")[-1]
GATEWAY_POLICY_NAME = "AnyCompanyGatewayPermissions"

existing = iam.get_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName=GATEWAY_POLICY_NAME)["PolicyDocument"]
for stmt in existing["Statement"]:
    if stmt.get("Sid") == "InvokeLambdaTargets":
        res = stmt["Resource"] if isinstance(stmt["Resource"], list) else [stmt["Resource"]]
        if INTERCEPTOR_LAMBDA_ARN not in res:
            res.append(INTERCEPTOR_LAMBDA_ARN); stmt["Resource"] = res
            print(f"Added {INTERCEPTOR_LAMBDA_ARN}")
        else:
            print("Already present.")
        break
else:
    raise RuntimeError("InvokeLambdaTargets statement not found.")

iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName=GATEWAY_POLICY_NAME, PolicyDocument=json.dumps(existing))
print("Gateway role updated.")
time.sleep(10)

Added arn:aws:lambda:us-west-2:359894365202:function:anycompany_aiguard_interceptor
Gateway role updated.


## Step 5: Attach the REQUEST Interceptor (keep the RESPONSE PII interceptor)

`update_gateway` replaces the entire `interceptorConfigurations` list and **cannot accept `[]`**. A gateway allows
**at most one REQUEST and one RESPONSE** interceptor, so we re-send the kept PII RESPONSE config alongside the new
AI Guard REQUEST config. We look up the PII interceptor ARN from SSM (or fall back to the live gateway config).

In [5]:
# The kept PII RESPONSE interceptor ARN — published by 1_sensitive_data_masking.ipynb (Step 5),
# with a fallback to the live gateway config. update_gateway REPLACES the whole interceptor list
# and rejects [], so we MUST re-send the RESPONSE config or PII masking is silently dropped.
PII_INTERCEPTOR_ARN = get_ssm(f"{SSM_PREFIX}/pii_interceptor_arn", default="")
gw = agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID)
if not PII_INTERCEPTOR_ARN:
    # Fallback: read it off the live gateway. NOTE: get_gateway does not echo
    # interceptorConfigurations on every SDK version, so this may come back empty even when a
    # RESPONSE interceptor IS attached — which is exactly why nb1 publishes the ARN to SSM.
    for cfg in gw.get("interceptorConfigurations", []):
        if "RESPONSE" in cfg.get("interceptionPoints", []):
            PII_INTERCEPTOR_ARN = cfg["interceptor"]["lambda"]["arn"]
            print("Resolved PII interceptor ARN from live gateway config (SSM key was absent).")

if not PII_INTERCEPTOR_ARN:
    raise RuntimeError(
        "Could not resolve the PII RESPONSE interceptor ARN from SSM "
        f"({SSM_PREFIX}/pii_interceptor_arn) or the live gateway config.\n"
        "Run l4-security/1_sensitive_data_masking.ipynb FIRST — it attaches the RESPONSE "
        "interceptor and publishes its ARN to SSM. Proceeding without it would REPLACE the "
        "interceptor list and silently drop PII masking. Failing loud instead of degrading security."
    )
print(f"Kept PII RESPONSE interceptor: {PII_INTERCEPTOR_ARN}")

interceptor_configs = [
    {
        "interceptor": {"lambda": {"arn": PII_INTERCEPTOR_ARN}},
        "interceptionPoints": ["RESPONSE"],
        "inputConfiguration": {"passRequestHeaders": True},
    },
    {
        "interceptor": {"lambda": {"arn": INTERCEPTOR_LAMBDA_ARN}},
        "interceptionPoints": ["REQUEST"],
        "inputConfiguration": {"passRequestHeaders": True},  # REQUIRED so we see Authorization
    },
]

agentcore_control.update_gateway(
    gatewayIdentifier=GATEWAY_ID,
    name=GATEWAY_NAME,
    roleArn=GATEWAY_ROLE_ARN,
    # update_gateway REPLACES the entire authorizer. Re-send the M2M config
    # (allowedClients + allowedScopes) — NOT allowedAudience — or attaching this
    # interceptor reverts L3's M2M auth and every tool call 403s (insufficient_scope).
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_USER_POOL_ID}/.well-known/openid-configuration",
        "allowedClients": [GATEWAY_M2M_CLIENT_ID], "allowedScopes": [GATEWAY_SCOPE]}},
    interceptorConfigurations=interceptor_configs,
)
print(f"Gateway updated with {len(interceptor_configs)} interceptor(s): RESPONSE (PII) + REQUEST (AI Guard breaker).")
for _ in range(30):
    if agentcore_control.get_gateway(gatewayIdentifier=GATEWAY_ID).get("status") == "READY":
        print("Gateway READY."); break
    time.sleep(5)

Kept PII RESPONSE interceptor: arn:aws:lambda:us-west-2:359894365202:function:anycompany-pii-interceptor
Gateway updated with 2 interceptor(s): RESPONSE (PII) + REQUEST (AI Guard breaker).
Gateway READY.


## Step 6: Verify the Circuit Breaker

1. **Seed a LOCK** for the service `sub` → a tool call returns **403** and the tool Lambda is **not** invoked.
2. **`sub` match** — confirm the `sub` the interceptor decodes equals the `sub` the L3 hook wrote.
3. **Invoke the chatbot or Harness** — confirm the response shows a Gateway error.

**How to verify in Datadog APM:**
- Filter by `service:anycompany-order-agent` — you should see **no new spans** during the lock period.
  This is proof the block is working: the Gateway interceptor rejects the request (403) before it
  reaches the agent container, so no ddtrace span is generated.
- The absence of new traces IS the evidence of successful enforcement.

**In CloudWatch (Gateway APPLICATION_LOGS):**
- The interceptor Lambda logs the 403 decision with the reason and `principalId`.


In [6]:
import boto3, json, base64
ddb = boto3.client("dynamodb", region_name=REGION)
import requests as req_lib

# The breaker keys on the `sub` of the M2M service token the agents present to the
# gateway — the SAME value the REQUEST interceptor decodes from the Authorization
# header. Mint that token the same way (client_credentials) so Step 6 seeds and
# Step 7 clears the REAL principal.
def _decode_jwt_sub(token):
    payload_b64 = token.split(".")[1]
    payload_b64 += "=" * (-len(payload_b64) % 4)  # pad to a multiple of 4
    return json.loads(base64.urlsafe_b64decode(payload_b64)).get("sub", "")

_m2m_id = get_ssm(f"{SSM_PREFIX}/gateway_m2m_client_id")
_m2m_secret = get_ssm(f"{SSM_PREFIX}/gateway_m2m_client_secret")
_tok_ep = get_ssm(f"{SSM_PREFIX}/gateway_token_endpoint")
_scope = get_ssm(f"{SSM_PREFIX}/gateway_scope")
_basic = base64.b64encode(f"{_m2m_id}:{_m2m_secret}".encode()).decode()
_tok = req_lib.post(_tok_ep, data={"grant_type": "client_credentials", "scope": _scope},
    headers={"Authorization": f"Basic {_basic}", "Content-Type": "application/x-www-form-urlencoded"}, timeout=30)
_tok.raise_for_status()
SERVICE_SUB = _decode_jwt_sub(_tok.json()["access_token"])
print(f"Service sub (breaker key): {SERVICE_SUB}")

# Guard: the 403 below only occurs if the gateway breaker is ENABLED. A prior
# Step 7b DISABLE writes an `enabled=false` config row that PERSISTS across sessions,
# and the interceptor then bypasses the lock lookup entirely - so this seed would NOT
# 403 and the demo would silently show a 200. Warn loudly before seeding.
_cfg = ddb.get_item(TableName=RISK_TABLE, Key={"principalId": {"S": "__breaker_config__"}},
                    ConsistentRead=True).get("Item")
if _cfg and _cfg.get("enabled", {}).get("BOOL", True) is False:
    print("\nWARNING: gateway breaker is GLOBALLY DISABLED (Step 7b enabled=false).")
    print("This test will NOT return 403 - run the Step 7b RE-ENABLE cell, then re-run this cell.\n")

# Seed a LOCK (mimics what the L3 hook writes on ABORT).
ttl = int(time.time()) + 900
ddb.put_item(TableName=RISK_TABLE, Item={
    "principalId": {"S": SERVICE_SUB}, "status": {"S": "LOCKED"},
    "reason": {"S": "manual seed for verification"},
    "ts": {"N": str(int(time.time()))}, "ttl": {"N": str(ttl)}})
print(f"Seeded LOCK for {SERVICE_SUB}. Invoking a tool call — expect 403, tool Lambda not invoked.\n")

# ── Verification 1: confirm the lock row is present in DynamoDB ──────────────
lock_item = ddb.get_item(
    TableName=RISK_TABLE,
    Key={"principalId": {"S": SERVICE_SUB}},
    ConsistentRead=True,
).get("Item")
if lock_item and lock_item.get("status", {}).get("S") == "LOCKED":
    print(f"✅ PASS (DynamoDB) — LOCK row confirmed for principalId={SERVICE_SUB}")
else:
    print(f"❌ FAIL (DynamoDB) — Lock row missing or not LOCKED. Item: {lock_item}")

# ── Verification 2: invoke the Harness and confirm the tool call was degraded ─
# The Harness (orchestrator) has no AI Guard hook, so it catches the 403 from the
# gateway and the LLM rephrases it as an "authorization/access issue" — it will NOT
# surface a raw 403. Do NOT parse the LLM response text; that is non-deterministic.
import uuid
from botocore.config import Config

agentcore_runtime = boto3.client("bedrock-agentcore", region_name=REGION, config=Config(read_timeout=60))
HARNESS_ARN = get_ssm(f"{SSM_PREFIX}/harness_arn")

# Record the time just before invoking so we can narrow the CWL search window.
invoke_start_ms = int(time.time() * 1000)

session_id = str(uuid.uuid4()).upper()
response_text = []

try:
    response = agentcore_runtime.invoke_harness(
        harnessArn=HARNESS_ARN,
        runtimeSessionId=session_id,
        actorId="CUST-789",
        messages=[{"role": "user", "content": [{"text": "What is the status of order ORD-10001?"}]}],
    )
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                response_text.append(delta["text"])
        elif "internalServerException" in event:
            print(f"✅ PASS (Harness stream) — internalServerException: {event['internalServerException']}")
except Exception as e:
    print(f"✅ PASS (Harness exception) — {e}")

if response_text:
    # The orchestrator LLM always rephrases the 403 in natural language ("authorization
    # issue", "access issue", etc.) — this is expected, not a failure.
    print(f"\nHarness response (LLM rephrased the 403 — expected behaviour):\n{''.join(response_text)}")

# ── Verification 3: pull the 403 log line directly from CloudWatch Logs ───────
# Wait a few seconds for the Lambda log to flush, then query the log stream.
print("\n── Verification 3: CloudWatch interceptor logs ──")
print("Waiting 5s for Lambda logs to flush...", end=" ", flush=True)
time.sleep(5)
print("done.")

logs = boto3.client("logs", region_name=REGION)
LOG_GROUP = f"/aws/lambda/{INTERCEPTOR_LAMBDA_NAME}"
found_403 = False

try:
    # Get the most recent log streams (sorted by last event time).
    streams = logs.describe_log_streams(
        logGroupName=LOG_GROUP,
        orderBy="LastEventTime",
        descending=True,
        limit=5,
    ).get("logStreams", [])

    for stream in streams:
        events = logs.get_log_events(
            logGroupName=LOG_GROUP,
            logStreamName=stream["logStreamName"],
            startTime=invoke_start_ms,
            startFromHead=True,
        ).get("events", [])

        for event in events:
            msg = event.get("message", "")
            if "circuit breaker OPEN" in msg and SERVICE_SUB in msg:
                print(f"✅ PASS (CloudWatch) — Interceptor fired 403:")
                print(f"   {msg.strip()}")
                found_403 = True
                break
        if found_403:
            break

    if not found_403:
        print("⚠️  No 403 log line found yet in the last 5 log streams since invocation.")
        print("   This can happen if logs haven't flushed or the Lambda cold-started.")
        print(f"   Check manually in CloudWatch Log Insights on {LOG_GROUP}:")
        print(f"     fields @timestamp, @message")
        print(f"     | filter @message like /circuit breaker OPEN/")
        print(f"     | filter @message like /{SERVICE_SUB}/")
        print(f"     | sort @timestamp desc | limit 5")
except Exception as e:
    print(f"   Could not query CloudWatch logs: {e}")
    print(f"   Check manually in CloudWatch Log Insights on {LOG_GROUP}.")

Service sub (breaker key): 77qsb5ks15masi1gqnsdl8op0e
Seeded LOCK for 77qsb5ks15masi1gqnsdl8op0e. Invoking a tool call — expect 403, tool Lambda not invoked.

✅ PASS (DynamoDB) — LOCK row confirmed for principalId=77qsb5ks15masi1gqnsdl8op0e

Harness response (LLM rephrased the 403 — expected behaviour):
I'll look up the order details for you. Let me search for the Order Agent and retrieve the information simultaneously.Found the Order Agent. Now delegating the lookup to it with your customer ID.I'm sorry, I was unable to retrieve the order details at this time — the Order Agent returned an access error (403). I cannot fabricate or estimate order information, so I'm unable to provide the status of ORD-10001 right now.

Here's what you can do:
- **Try again in a moment** — this may be a temporary service issue.
- **Contact support directly** if the issue persists, and reference your order number **ORD-10001** and customer ID **CUST-789**.

Is there anything else I can help you with?

── 

## Step 7: Restore Normal Operation

Delete the test LOCK row to restore normal tool call routing. Without this, all tool
calls remain blocked (403) until the TTL expires (15 minutes).

> **Step 7 vs Step 7b:** this clears a single test LOCK but leaves the breaker **active**
> (a new ABORT will re-lock). To disable enforcement entirely for testing, use **Step 7b**,
> which also clears the lock as a convenience.

**After clearing the lock:** proceed to `5_aiguard_enforcement_tests.ipynb` (Notebook 4)
which invokes the Harness with test scenarios. This generates fresh traces in Datadog APM.

**How to verify in Datadog APM:**
- After running Notebook 4, filter by `service:anycompany-order-agent`
- **New spans appear** within seconds — the trace shows Agent → Gateway → Lambda (normal 200)
- Compare the timestamp gap: no spans during the lock, then traces resume after clearing


In [7]:
# Delete the test LOCK row to restore normal operation.
ddb.delete_item(TableName=RISK_TABLE, Key={"principalId": {"S": SERVICE_SUB}})
print(f"Lock cleared for {SERVICE_SUB} — tool calls will succeed again.")
print()
print("Verify: invoke the chatbot or Harness — tool calls should return 200.")
print("In Datadog APM: new traces show normal Gateway → Lambda spans (no 403).")


Lock cleared for 77qsb5ks15masi1gqnsdl8op0e — tool calls will succeed again.

Verify: invoke the chatbot or Harness — tool calls should return 200.
In Datadog APM: new traces show normal Gateway → Lambda spans (no 403).


## Step 7b: Toggle the Gateway Breaker for Testing (real-time, no redeploy)

For workshop testing you can globally disable the gateway circuit breaker **without
redeploying** anything. The interceptor reads a single on/off config row from the
**same risk table** on every tool call (strongly consistent, no cache), so a flip
takes effect on the very next call.

- **Disable** (next cell): writes `enabled=false` and clears any existing lock, so
  tool calls route with **no 403** — you can exercise the agents without the breaker.
- **Re-enable**: writes `enabled=true` (or delete the row — it fails safe to ENABLED).

> **Fail-safe:** a missing or errored config row means ENABLED, so the breaker can
> only be turned off by an explicit `enabled=false` write — never by deletion, absence,
> or a transient read error.
>
> **Production note:** this is a test/ops kill-switch. It rides on the *same* restricted
> risk-table write access the lock already uses, so it adds no new writer here. But it is
> a **fail-open** control: if you later broaden who can write the risk table, isolate this
> config (separate table/IAM) and alarm when it flips to disabled.

In [8]:
# --- DISABLE the gateway breaker for testing (real-time) ---
# Writes the global on/off config row (enabled=false) AND clears any lingering lock,
# so tool calls route with no 403. The interceptor picks this up on the NEXT tool call
# (strongly consistent read - no cache, no redeploy).
import time as _t
BREAKER_CONFIG_KEY = "__breaker_config__"

ddb.put_item(TableName=RISK_TABLE, Item={
    "principalId": {"S": BREAKER_CONFIG_KEY},
    "enabled": {"BOOL": False},
    "note": {"S": "workshop testing - breaker disabled"},
    "ts": {"N": str(int(_t.time()))},
})
print("Gateway breaker DISABLED (enabled=false) - tool calls pass through, no 403.")

# Clear any existing lock for the service principal so nothing 403s while disabled.
try:
    ddb.delete_item(TableName=RISK_TABLE, Key={"principalId": {"S": SERVICE_SUB}})
    print(f"Cleared lock for {SERVICE_SUB}.")
except NameError:
    print("(SERVICE_SUB not in this session - run Step 6 first if a lock needs clearing.)")

print("Re-run the ENABLE cell below to restore enforcement.")

Gateway breaker DISABLED (enabled=false) - tool calls pass through, no 403.
Cleared lock for 77qsb5ks15masi1gqnsdl8op0e.
Re-run the ENABLE cell below to restore enforcement.


Re-enable the breaker to restore normal enforcement (run when you are done testing).

In [9]:
# --- RE-ENABLE the gateway breaker (restore enforcement) ---
import time as _t
BREAKER_CONFIG_KEY = "__breaker_config__"

ddb.put_item(TableName=RISK_TABLE, Item={
    "principalId": {"S": BREAKER_CONFIG_KEY},
    "enabled": {"BOOL": True},
    "ts": {"N": str(int(_t.time()))},
})
print("Gateway breaker RE-ENABLED (enabled=true) - lock lookups resume on the next tool call.")
print("(Equivalent: delete the config row - the interceptor fails safe to ENABLED.)")

Gateway breaker RE-ENABLED (enabled=true) - lock lookups resume on the next tool call.
(Equivalent: delete the config row - the interceptor fails safe to ENABLED.)


## Step 8: Generate AI Guard Evaluations

With the lock cleared (Step 7), invoke the Harness a couple of times so the Order and
Refund agents actually run. Because AI Guard was enabled on those agents in
`2_aiguard_agent_protection.ipynb` (`AI_GUARD_ENABLED=true`), every model and tool
checkpoint calls AI Guard's `evaluate()` — and those calls surface in
**Datadog → AI Guard → Investigate**.

The benign prompts below produce **ALLOW** verdicts; the adversarial scenarios in
`5_aiguard_enforcement_tests.ipynb` produce **DENY/ABORT** verdicts. If the Investigate
tab shows zero evaluations, this is the quickest way to confirm the hooks are firing
(and that `DD_APP_KEY` carries the `ai_guard` scope).

> Requires the Harness from `l3-orchestration/4_orchestrator_agent.ipynb` (`harness_arn` in SSM).


In [10]:
# Invoke the Harness so the AI-Guard-enabled agents run and call evaluate().
# Run this AFTER clearing the lock (Step 7) so tool calls succeed end-to-end.
import uuid
from botocore.config import Config

agentcore_runtime = boto3.client("bedrock-agentcore", region_name=REGION, config=Config(read_timeout=300))
HARNESS_ARN = get_ssm(f"{SSM_PREFIX}/harness_arn")
ACTOR_ID = "CUST-789"

def invoke_harness(prompt, tier="gold"):
    session_id = str(uuid.uuid4()).upper()
    print(f"\n>>> [{tier}] {prompt}")
    response = agentcore_runtime.invoke_harness(
        harnessArn=HARNESS_ARN,
        runtimeSessionId=session_id,
        actorId=ACTOR_ID,
        messages=[{"role": "user", "content": [{"text": f"[Authenticated customer: {ACTOR_ID} | Tier: {tier}]\n\n{prompt}"}]}],
    )
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
        elif "messageStop" in event:
            print()
        elif "internalServerException" in event:
            print(f"\nError: {event['internalServerException']}")

# Two benign multi-agent prompts — each drives the Order/Refund agents, so AI Guard
# evaluates several model/tool checkpoints (all ALLOW verdicts).
invoke_harness("What is the status of order ORD-10001?")
invoke_harness("Which of my orders are eligible for a refund?")

print("\n\nDone. Open Datadog → AI Guard → Investigate (last 15 min) — you should now see")
print("evaluations for the order/refund agents. Adversarial DENY/ABORT verdicts come from")
print("5_aiguard_enforcement_tests.ipynb.")



>>> [gold] What is the status of order ORD-10001?
I'll look up the order details for you. Let me search for the Order Agent and retrieve the information simultaneously.

Found the Order Agent. Now delegating the lookup to it, scoped to your account.

I'm sorry, I was unable to retrieve the details for order ORD-10001 at this time. The order lookup service is currently unavailable (access was blocked). I cannot provide order details, status, or any related information without a successful response from the system.

Please try again in a few moments, or contact our support team directly if the issue persists. I apologize for the inconvenience! 🙏

>>> [gold] Which of my orders are eligible for a refund?
I'll help you check refund eligibility for your orders! Let me start by finding the right specialist agents and retrieving your orders simultaneously.

**Step 1: Searching for specialist agents and fetching your orders...**

Found both agents! Now let me fetch your full order history.

**

## Summary

| Component | Resource | IAM |
|---|---|---|
| REQUEST interceptor | `anycompany_aiguard_interceptor` Lambda | own role: `dynamodb:GetItem` on risk table |
| Gateway attach | `update_gateway` REQUEST + kept RESPONSE | Gateway role: `lambda:InvokeFunction` on interceptor |
| SSM | `/anycompany/agentcore/aiguard_interceptor_arn` | — |
| Breaker kill-switch (testing) | `__breaker_config__` row in the risk table (`enabled` bool), read live by the interceptor | none new (reuses interceptor `dynamodb:GetItem`) |

**The breaker is service-wide.** It keys on the fixed service `sub`, so one `ABORT` 403s every tool call until TTL.
That is expected given the single shared service identity — see the production-hardening callout below.

> ### 🏭 Production hardening — this breaker does NOT scale to multi-user
> The lock is keyed on the **fixed service identity** the gateway sees, because this topology has **no deterministic
> per-user/per-session id at the gateway**: agents authenticate with one shared service token, the harness
> `runtimeSessionId` does not propagate to specialists, and harness→specialist delegation is LLM-mediated. So one
> user's `ABORT` would lock **everyone**. In production, scope the lock per-user via **AgentCore Identity
> act-on-behalf**, deriving the key from a *verified* per-user claim so containment is isolated to the offending
> session. Build the breaker as shown; surface the gap honestly as the architecture lesson.

## Cleanup

⚠️ Commented out. **Order:** run this BEFORE the L3 `1_setup_resources.ipynb` cleanup (which owns the Gateway/role),
and detach the REQUEST interceptor by re-sending ONLY the kept RESPONSE config (you cannot pass `[]`).

In [11]:
## === CLEANUP: AI Guard Gateway circuit breaker ===
## Uncomment and run only after you are done with ALL notebooks. Run BEFORE l3 1_setup_resources cleanup.
#
# import boto3, os, json, time
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
# INTERCEPTOR_LAMBDA_NAME = "anycompany_aiguard_interceptor"
# INTERCEPTOR_ROLE_NAME   = "AnyCompanyAIGuardInterceptorLambdaRole"
#
# ssm = boto3.client("ssm", region_name=REGION)
# iam = boto3.client("iam")
# lambda_client = boto3.client("lambda", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
#
# def _ssm(k, d=None):
#     try: return ssm.get_parameter(Name=f"{SSM_PREFIX}/{k}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound: return d
#
# # 1. Detach REQUEST interceptor: re-send ONLY the kept RESPONSE config (cannot pass []).
# gid=_ssm("gateway_id"); gname=_ssm("gateway_name"); grole=_ssm("gateway_role_arn")
# pool=_ssm("cognito_user_pool_id"); client=_ssm("cognito_client_id"); pii=_ssm("pii_interceptor_arn")
# m2m=_ssm("gateway_m2m_client_id"); gscope=_ssm("gateway_scope")
# kept = []
# if pii:
#     kept.append({"interceptor": {"lambda": {"arn": pii}}, "interceptionPoints": ["RESPONSE"],
#                  "inputConfiguration": {"passRequestHeaders": True}})
# if gid and kept:
#     agentcore_control.update_gateway(gatewayIdentifier=gid, name=gname, roleArn=grole,
#         authorizerType="CUSTOM_JWT", authorizerConfiguration={"customJWTAuthorizer": {
#             "discoveryUrl": f"https://cognito-idp.{REGION}.amazonaws.com/{pool}/.well-known/openid-configuration",
#             "allowedClients": [m2m], "allowedScopes": [gscope]}}, interceptorConfigurations=kept)
#     print("Detached REQUEST interceptor (kept RESPONSE).")
#
# # 2. Revert the Gateway-role InvokeLambdaTargets append.
# arn = _ssm("aiguard_interceptor_arn")
# if arn and grole:
#     rn = grole.split("/")[-1]; pol = iam.get_role_policy(RoleName=rn, PolicyName="AnyCompanyGatewayPermissions")["PolicyDocument"]
#     for s in pol["Statement"]:
#         if s.get("Sid")=="InvokeLambdaTargets":
#             res=[r for r in (s["Resource"] if isinstance(s["Resource"],list) else [s["Resource"]]) if r!=arn]
#             s["Resource"]=res
#     iam.put_role_policy(RoleName=rn, PolicyName="AnyCompanyGatewayPermissions", PolicyDocument=json.dumps(pol))
#
# # 3. Delete the Lambda + its role + SSM key.
# try: lambda_client.delete_function(FunctionName=INTERCEPTOR_LAMBDA_NAME)
# except Exception as e: print(e)
# try:
#     iam.delete_role_policy(RoleName=INTERCEPTOR_ROLE_NAME, PolicyName="AIGuardInterceptorPerms")
#     iam.delete_role(RoleName=INTERCEPTOR_ROLE_NAME)
# except Exception as e: print(e)
# try: ssm.delete_parameter(Name=f"{SSM_PREFIX}/aiguard_interceptor_arn")
# except ssm.exceptions.ParameterNotFound: pass
# print("\n✅ AI Guard gateway breaker cleaned up.")